# Python Regular Expressions (`re`): Master Text Pattern Processing

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/plus2net/Python-basics/blob/main/python_string_re_practice.ipynb)

Welcome to the interactive companion notebook for the **[plus2net Python Regular Expressions Tutorial](https://www.plus2net.com/python/string-re.php)**.

### Learning Objectives:
1. Understand why raw string literals `r"..."` are essential in regular expressions.
2. Master the scan anchor differences between `search()`, `match()`, and `fullmatch()`.
3. Extract structured data using positional and named capturing groups.
4. Perform mass pattern extractions with `findall()` and substitutions with `sub()`.
5. Precompile reusable patterns using `re.compile()` with regex flags.
6. Validate implementations with automated self-checking unit assertions.

## 1. Comparing Scan Anchors: `search()` vs. `match()` vs. `fullmatch()`

- `match()` matches only at the *beginning* of the string.
- `search()` scans the *entire* string for the first match.
- `fullmatch()` requires the *entire* string to conform to the pattern.

In [ ]:
import re

text = "Error code: 404 in server logs"

# 1. match() fails because '404' is not at index 0
assert re.match(r"\d+", text) is None

# 2. search() finds '404'
m = re.search(r"\d+", text)
assert m is not None and m.group() == "404"

# 3. fullmatch() requires entire string match
assert re.fullmatch(r"\d+", text) is None
assert re.fullmatch(r"\d+", "404") is not None

print("Scan anchor assertions passed!")


## 2. Capturing Groups: Positional & Named

Parentheses define capturing groups. Named groups `(?P<name>...)` allow dictionary extraction via `groupdict()`.

In [ ]:
log_line = "2026-10-14 [ERROR] Database timeout on host db.local:5432"
pattern = r"(?P<date>\d{4}-\d{2}-\d{2})\s+\[(?P<level>[A-Z]+)\]\s+(?P<msg>.+)"

match = re.search(pattern, log_line)
assert match is not None

data = match.groupdict()
print("Parsed log data:", data)
assert data["date"] == "2026-10-14"
assert data["level"] == "ERROR"
assert data["msg"] == "Database timeout on host db.local:5432"

print("Capturing group assertions passed!")


## 3. Mass Extraction & Masking: `findall()` & `sub()`

- `findall()` returns all non-overlapping matches.
- `sub()` replaces occurrences using string templates or backreferences.

In [ ]:
corpus = "Birthdays: Alice (24-12-2002), Bob (05-09-1998), Charlie (12-04-2005)"

# 1. Extract dates
dates = re.findall(r"\b\d{2}-\d{2}-\d{4}\b", corpus)
print("Extracted dates:", dates)
assert dates == ["24-12-2002", "05-09-1998", "12-04-2005"]

# 2. Mask years with XXXX
masked = re.sub(r"(\d{2}-\d{2}-)\d{4}", r"\g<1>XXXX", corpus)
print("Masked text:    ", masked)
assert "2002" not in masked and "XXXX" in masked

print("\nMass extraction and masking assertions passed!")


## 4. Precompilation & Flags: `re.compile()`

Precompiling patterns saves execution time in loops. Flags like `re.IGNORECASE` enable case-insensitive matching.

In [ ]:
EMAIL_PATTERN = re.compile(r"^[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+$", re.IGNORECASE)

assert bool(EMAIL_PATTERN.match("admin@plus2net.com")) is True
assert bool(EMAIL_PATTERN.match("ADMIN@PLUS2NET.COM")) is True
assert bool(EMAIL_PATTERN.match("invalid_email")) is False

print("Precompiled email validation passed!")


## 5. Real-World Engineering Recipes

Explore three common software engineering patterns utilizing regular expressions:
1. Date Format Normalization (DD-MM-YYYY to ISO YYYY-MM-DD)
2. HTML Hyperlink URL Extractor
3. Multi-Delimiter String Splitting

In [ ]:
# Recipe 1: Date reformatting
def to_iso_date(text: str) -> str:
    return re.sub(r"\b(\d{2})-(\d{2})-(\d{4})\b", r"\3-\2-\1", text)

raw_doc = "Release date is 24-12-2026."
assert to_iso_date(raw_doc) == "Release date is 2026-12-24."

# Recipe 2: Hyperlink extraction
def get_links(html_text: str) -> list[str]:
    return re.findall(r'''href=['"]?([^'" >]+)''', html_text, re.IGNORECASE)

sample_html = '<a href="https://plus2net.com">Home</a><a href=/docs>Docs</a>'
links = get_links(sample_html)
print("Extracted links:", links)
assert links == ["https://plus2net.com", "/docs"]

# Recipe 3: Multi-delimiter splitting
def split_tokens(raw: str) -> list[str]:
    return [t for t in re.split(r"[,;| \t\n]+", raw) if t]

tokens = split_tokens("alpha, beta; gamma|delta   epsilon")
assert tokens == ["alpha", "beta", "gamma", "delta", "epsilon"]

print("\nAll engineering recipes passed!")


## 6. Production Implementation & Self-Checking Unit Assertions

This production-grade `RegexParser` class encapsulates date extraction, ISO reformatting, URL scraping, and automated unit assertions.

In [ ]:
class RegexParser:
    DATE_PATTERN = re.compile(r"\b(\d{2})-(\d{2})-(\d{4})\b")
    URL_PATTERN = re.compile(r'''href=['"]?([^'" >]+)''', re.IGNORECASE)

    @classmethod
    def extract_dates(cls, text: str) -> list[str]:
        """Extracts date strings in DD-MM-YYYY format."""
        return [f"{d}-{m}-{y}" for d, m, y in cls.DATE_PATTERN.findall(text)]

    @classmethod
    def to_iso_dates(cls, text: str) -> str:
        """Converts DD-MM-YYYY dates to ISO YYYY-MM-DD format."""
        return cls.DATE_PATTERN.sub(r"\3-\2-\1", text)

    @classmethod
    def extract_hrefs(cls, html_str: str) -> list[str]:
        """Finds all URL targets in HTML markup."""
        return cls.URL_PATTERN.findall(html_str)


# ==========================================
# UNIT ASSERTION SUITE
# ==========================================
parser = RegexParser()

# 1. Date extraction
sample_text = "Event on 24-12-2002 and followup on 15-01-2003"
dates = parser.extract_dates(sample_text)
assert dates == ["24-12-2002", "15-01-2003"]

# 2. ISO Date reformatting
iso_text = parser.to_iso_dates(sample_text)
assert iso_text == "Event on 2002-12-24 and followup on 2003-01-15"

# 3. URL extraction
html_doc = '<a href="https://plus2net.com">Home</a><a href=\'/python/string.php\'>Strings</a>'
urls = parser.extract_hrefs(html_doc)
assert urls == ["https://plus2net.com", "/python/string.php"]

# 4. Search vs Match semantics
assert re.match(r"\d+", "abc 123") is None
assert re.search(r"\d+", "abc 123").group() == "123"

# 5. Non-greedy quantifier check
html_snippet = "<b>first</b> and <b>second</b>"
assert re.findall(r"<b>(.*?)</b>", html_snippet) == ["first", "second"]

print("All regular expression production unit assertions passed successfully!")
